# Part 4: Micro-Agents Inference & Pipeline Orchestration (Local Execution)

This notebook covers:
- Initializing crawler, scorer, divergence detector, and aggregator micro-agents
- Streaming and scoring sentiment across test-period StockNet data
- Detecting price-sentiment divergence anomalies
- Generating trade and sentiment divergence signals

In [1]:
import sys
sys.path.append("..")

from config.global_config import CONFIG
from src.agents.sentiment_scorer import SentimentScorerAgent
import pandas as pd
import json
from sklearn.metrics import accuracy_score, f1_score
import numpy as np

scorer_agent = SentimentScorerAgent(model_checkpoint=CONFIG["checkpoint_dir"])

test_df = pd.read_parquet(f"{CONFIG['processed_dir']}/test.parquet")
print(f"Loaded {len(test_df)} test sentences")

Loaded 727 test sentences


# Run inference through the agent and compute metrics

In [2]:
LABEL2ID = {"Negative": 0, "Neutral": 1, "Positive": 2}

results = scorer_agent.score_batch(
    test_df["sentence"].tolist(),
    batch_size=CONFIG["train_batch_size"],
    max_seq_len=CONFIG["max_seq_len"],
)

y_pred = np.array([LABEL2ID[r["label"]] for r in results])
y_true = test_df["label"].to_numpy()

pipeline_metrics = {
    "accuracy": float(accuracy_score(y_true, y_pred)),
    "macro_f1": float(f1_score(y_true, y_pred, average="macro", zero_division=0)),
}
class_f1s = f1_score(y_true, y_pred, average=None, zero_division=0)
for idx, name in enumerate(["negative", "neutral", "positive"]):
    pipeline_metrics[f"f1_{name}"] = float(class_f1s[idx])

print(pipeline_metrics)

{'accuracy': 0.8528198074277854, 'macro_f1': 0.8329816526171153, 'f1_negative': 0.8177339901477833, 'f1_neutral': 0.8883720930232558, 'f1_positive': 0.7928388746803069}


# Save Result

In [3]:
out_path = f"{CONFIG.get('results_dir', '../results')}/metrics/pipeline_test_metrics.json"
# CONFIG has no "results_dir" key — use the relative path directly instead:
out_path = "../results/metrics/pipeline_test_metrics.json"

with open(out_path, "w") as f:
    json.dump(pipeline_metrics, f, indent=2)

print(f"Saved to {out_path}")

Saved to ../results/metrics/pipeline_test_metrics.json
